# 4 · Two modes of domain variation are set by exon architecture

Reproduces the scatter of **fraction skipped (among altered isoforms)** vs
**median exon count per Pfam family**, with the marginal skipped/trimmed bars.
Single-exon modules toggle whole (top-left); multi-exon cores get trimmed
(bottom-right). Built entirely from raw files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# raw tables (built by scripts 01-02)
RAW = Path('/data1/peerd/sotougl/fastcds/data/raw')

In [ ]:
# ---------- one row per domain, from the master + intervals ----------
# label every domain-isoform comparison
m = pd.read_csv(RAW / 'domain_isoform_master.tsv', sep='\t',
                usecols=['domain_id', 'pfam_id', 'clan_name',
                         'domain_bp', 'covered_bp', 'conservation_class'])
cover = (m.covered_bp / m.domain_bp.replace(0, np.nan)).clip(0, 1)
m['geom'] = np.where(m.conservation_class == 'conserved', 'intact',
            np.where(cover < 0.15, 'skipped', 'trimmed'))

# per domain: identity, isoform count, and intactness (fraction kept whole)
grp = m.groupby('domain_id')
dom = pd.DataFrame({
    'pfam_id': grp.pfam_id.first(),
    'clan_name': grp.clan_name.first(),
    'n_isoform_pairs': grp.size(),
    'frac_conserved': grp.geom.apply(lambda s: (s == 'intact').mean()),
}).reset_index()

# skipped fraction among a domain's altered (non-intact) comparisons
alt = m[m.geom != 'intact']
g = alt.groupby('domain_id').geom.value_counts().unstack(fill_value=0)
for c in ['skipped', 'trimmed']:
    if c not in g: g[c] = 0
g['n_altered'] = g.skipped + g.trimmed
g['excision_frac'] = g.skipped / g.n_altered
dom = dom.merge(g[['n_altered', 'excision_frac']], on='domain_id', how='left')

# exon count = number of distinct CDS exons a domain spans
gi = pd.read_csv(RAW / 'domain_genomic_intervals.tsv', sep='\t',
                 usecols=['domain_id', 'exon_number'])
n_exons = gi.groupby('domain_id').exon_number.nunique().rename('n_exons')
dom = dom.merge(n_exons, on='domain_id', how='left')
dom['single_exon'] = dom.n_exons == 1

# variable = kept whole in 10-90 % of its isoform comparisons
dom['variable'] = dom.frac_conserved.between(0.10, 0.90)
print(f'{len(dom):,} domains; {int(dom.variable.sum()):,} variable')

In [ ]:
# ---------- aggregate variable domains to the Pfam-family level ----------
var = dom[dom.variable]
fam = (var.groupby('pfam_id')
          .agg(clan=('clan_name', 'first'),
               n_var=('domain_id', 'size'),
               frac_intact=('frac_conserved', 'mean'),
               skip_frac=('excision_frac', 'mean'),
               med_exons=('n_exons', 'median'))
          .reset_index())
fam = fam[fam.n_var >= 15]                       # families with >=15 variable domains
rho, p = spearmanr(fam.med_exons, fam.skip_frac)
print(f'{len(fam)} families   Spearman rho = {rho:.2f}, p = {p:.1e}')

In [ ]:
# ---------- marginal bars: skipped/trimmed fraction by exact exon count ----------
vv = var.copy()
vv['ebin'] = np.minimum(vv.n_exons.fillna(0).astype(int), 10)   # cap at 10+
bins = (vv[vv.ebin >= 1].groupby('ebin')
          .agg(skip=('excision_frac', 'mean'), ndom=('domain_id', 'size'))
          .reindex(range(1, 11)))
bins['trim'] = 1 - bins.skip
print(bins.round(2))

In [ ]:
# clans to annotate (label -> clan_name in the data)
LABELS = {'C2H2-zf': 'C2H2-zf', 'SOCS': 'SOCS_box', 'BPTI': 'BPTI',
          'Viral_Gag': 'Viral_Gag', 'Rossmann': 'NADP_Rossmann',
          'MFS transporter': 'MFS', 'actin-fold ATPase': 'Actin_ATPase'}

fig = plt.figure(figsize=(9, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 1, height_ratios=[0.32, 1.0])
axb = fig.add_subplot(gs[0])                       # marginal bars
axs = fig.add_subplot(gs[1], sharex=axb)           # scatter

# --- top: stacked skipped (red) + trimmed (yellow) per exon count ---
x = np.arange(1, 11)
w = x * 0.085                                      # thin bars, even on a log axis
axb.bar(x, bins.skip, width=w, color='#d73027', label='skipped')
axb.bar(x, bins.trim, width=w, bottom=bins.skip, color='#fee08b', label='trimmed')
for xi, n in zip(x, bins.ndom):
    if np.isfinite(n):
        axb.text(xi, 1.03, f'{int(n):,}', rotation=90, ha='center', va='bottom', fontsize=6)
axb.set_ylim(0, 1.35); axb.set_yticks([0, 0.5, 1])
axb.set_ylabel('fraction of\naltered isoforms')
axb.legend(fontsize=8, loc='upper right'); axb.tick_params(labelbottom=False)
axb.set_title('Fractions of altered isoforms  (n domains above each bar)')

# --- bottom: one point per family ---
sz = fam.n_var * 0.20 + 10
sc = axs.scatter(fam.med_exons, fam.skip_frac, s=sz, c=fam.frac_intact,
                 cmap='viridis', vmin=0.50, vmax=0.85,
                 edgecolor='k', linewidth=0.3, alpha=0.85)
axs.set_xscale('log')
axs.set_xticks(x); axs.set_xticklabels(x); axs.minorticks_off()
axs.set_xlabel('Median number of exons per family')
axs.set_ylabel('Fraction of skipped domains\namong altered domains')
axs.text(0.62, 0.9, f'r = {rho:.2f}, p-val < 0.01', transform=axs.transAxes, fontsize=11)
cb = fig.colorbar(sc, ax=axs, fraction=0.045, pad=0.02)
cb.set_label('fraction of intact domains')

# annotate a few representative families (largest family in each named clan)
for label, clan in LABELS.items():
    sub = fam[fam.clan == clan]
    if len(sub):
        r = sub.loc[sub.n_var.idxmax()]
        axs.annotate(label, (r.med_exons, r.skip_frac),
                     xytext=(6, 6), textcoords='offset points', fontsize=8)

# size legend (number of domains per family)
for nv in (100, 1000):
    axs.scatter([], [], s=nv * 0.20 + 10, c='0.6', edgecolor='k',
                linewidth=0.3, label=f'{nv}')
axs.legend(title='domains per family', loc='lower left', labelspacing=1.4,
           borderpad=1.0, fontsize=8, title_fontsize=8, frameon=True)

plt.show()